In [19]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ----- Load dataset -----
df = pd.read_csv("LoanDataset.csv")

# ----- Create Total_Income -----
df['Total_Income'] = df['Applicant_Income'] + df['CoApplicant_Income']

# Loan Amount Bin
bins_loan = [50000, 100000, 150000, 200000, 250000, 300000]
labels_loan = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
df['Loan_Bin'] = pd.cut(df['Loan_Amount'], bins=bins_loan, labels=labels_loan)

# Total Income Bin
bins_income = [0, 30000, 60000, 90000, 120000, 150000]
labels_income = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
df['Income_Bin'] = pd.cut(df['Total_Income'], bins=bins_income, labels=labels_income)

# ----- Encode categorical variables with separate encoders -----
encoders = {}
categorical_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le  # store encoder for later use

# ----- Features & Target -----
X = df[['Loan_Amount', 'CoApplicant_Income', 'Total_Income', 'Applicant_Income',
        'Dependents', 'Property_Area', 'Self_Employed', 'Married', 'Gender', 'Credit_History']]
y = df['Loan_Status']

# ----- Train-Test Split -----
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ----- Train Logistic Regression -----
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# ----- Predict on Test Data -----
y_pred = model.predict(X_test)

# ----- Evaluation -----
print("Model Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ==============================
# 🔹 Predict for a New Applicant
# ==============================
print("\n--- New Applicant Loan Prediction ---")
loan_amount = float(input("Enter Loan Amount: "))
co_income = float(input("Enter Co-Applicant Income: "))
app_income = float(input("Enter Applicant Income: "))
total_income = app_income + co_income
dependents = int(input("Enter Number of Dependents: "))
property_area = input("Enter Property Area (Urban/Rural/Semiurban): ")
self_emp = input("Self Employed? (Yes/No): ")
married = input("Married? (Yes/No): ")
gender = input("Gender (Male/Female): ")
credit_history = int(input("Credit History (1=Good, 0=Bad): "))

# Encode using previously fitted encoders
property_area_enc = encoders['Property_Area'].transform([property_area])[0]
self_emp_enc = encoders['Self_Employed'].transform([self_emp])[0]
married_enc = encoders['Married'].transform([married])[0]
gender_enc = encoders['Gender'].transform([gender])[0]

# Create new applicant DataFrame
new_applicant = pd.DataFrame([[loan_amount, co_income, total_income, app_income,
                               dependents, property_area_enc, self_emp_enc,
                               married_enc, gender_enc, credit_history]],
                             columns=['Loan_Amount', 'CoApplicant_Income', 'Total_Income', 'Applicant_Income',
                                      'Dependents', 'Property_Area', 'Self_Employed', 'Married', 'Gender', 'Credit_History'])

# Predict probabilities
probabilities = model.predict_proba(new_applicant)[0]  # [P(Rejected), P(Approved)]
prob_rejected = probabilities[0]
prob_approved = probabilities[1]

# Prediction result
prediction = model.predict(new_applicant)[0]
status = "Approved" if prediction == 1 else "Rejected"

print(f"\nLoan Status Prediction: {status}")
print(f"Probability of Rejection: {prob_rejected:.2f}")
print(f"Probability of Approval: {prob_approved:.2f}")


Model Accuracy: 0.45

Confusion Matrix:
 [[6 4]
 [7 3]]

Classification Report:
               precision    recall  f1-score   support

           0       0.46      0.60      0.52        10
           1       0.43      0.30      0.35        10

    accuracy                           0.45        20
   macro avg       0.45      0.45      0.44        20
weighted avg       0.45      0.45      0.44        20


--- New Applicant Loan Prediction ---


Enter Loan Amount:  60000
Enter Co-Applicant Income:  45000
Enter Applicant Income:  40000
Enter Number of Dependents:  3
Enter Property Area (Urban/Rural/Semiurban):  Rural
Self Employed? (Yes/No):  Yes
Married? (Yes/No):  No
Gender (Male/Female):  Female
Credit History (1=Good, 0=Bad):  1



Loan Status Prediction: Rejected
Probability of Rejection: 0.53
Probability of Approval: 0.47


C:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
C:\Users\HP\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
